In [1]:
import torch
import torch.nn as nn
import numpy as np
import os, glob, librosa, joblib
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# ==========================================================================
# FIND LATEST RUN & LOAD ARCHITECTURE
# ==========================================================================
runs = sorted(glob.glob(os.path.join("experiments", "runs", "run_*")))
LATEST_RUN_DIR = runs[-1] if runs else None
MODELS_DIR = os.path.join("experiments", "models")
RAW_DIR = os.path.join("data", "raw")
PROCESSED_DIR = os.path.join("data", "processed")

print(f"Saving visuals to: {LATEST_RUN_DIR}")

# (Re-paste the CNNAutoencoder class here so it works independently)
class CNNAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self._conv_block(1, 32)
        self.enc2 = self._conv_block(32, 64)
        self.enc3 = self._conv_block(64, 128)
        self.enc4 = self._conv_block(128, 128)
        self.flatten = nn.Flatten()
        self.bottleneck_encode = nn.Linear(128 * 8 * 4, 64) 
        self.bottleneck_decode = nn.Linear(64, 128 * 8 * 4) 
        self.dec1 = self._deconv_block(128, 128)
        self.dec2 = self._deconv_block(128, 64)
        self.dec3 = self._deconv_block(64, 32)
        self.dec4 = nn.Sequential(nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1), nn.Sigmoid())

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(out_c), nn.LeakyReLU(0.2), nn.MaxPool2d(kernel_size=2, stride=2), nn.Dropout2d(0.2))

    def _deconv_block(self, in_c, out_c):
        return nn.Sequential(nn.ConvTranspose2d(in_c, out_c, kernel_size=3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(out_c), nn.LeakyReLU(0.2))

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3)
        flat = self.flatten(e4)
        compressed = self.bottleneck_encode(flat)
        d_input = self.bottleneck_decode(compressed).view(-1, 128, 8, 4)
        return self.dec4(self.dec3(self.dec2(self.dec1(d_input))))

def compute_pauc(y_true, y_score, max_fpr=0.1):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    stop_idx = np.searchsorted(fpr, max_fpr, side='right')
    if stop_idx == 0: return 0.0
    fpr_truncated = fpr[:stop_idx]
    tpr_truncated = tpr[:stop_idx]
    if fpr_truncated[-1] < max_fpr:
        tpr_interp = np.interp(max_fpr, fpr, tpr)
        fpr_truncated = np.append(fpr_truncated, max_fpr)
        tpr_truncated = np.append(tpr_truncated, tpr_interp)
    return np.trapezoid(tpr_truncated, fpr_truncated) / max_fpr

# ==========================================================================
# EVALUATION LOOP
# ==========================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
machines = sorted([d for d in os.listdir(PROCESSED_DIR) if os.path.isdir(os.path.join(PROCESSED_DIR, d))])
criterion = nn.L1Loss(reduction='none')

for machine in machines:
    print(f"\nEVALUATING: {machine}")
    model_path = os.path.join(MODELS_DIR, f"best_{machine}.pth")
    scaler_path = os.path.join(PROCESSED_DIR, machine, "scaler.save")
    
    if not os.path.exists(model_path): continue
        
    model = CNNAutoencoder().to(device)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()
    
    scaler = joblib.load(scaler_path)
    test_files = glob.glob(os.path.join(RAW_DIR, machine, "test", "*.wav"))
    
    labels, scores = [], []
    saved_visual = False
    
    for fpath in test_files:
        is_anomaly = 1 if 'anomaly' in os.path.basename(fpath).lower() else 0
        
        # NOTE: Keeping Mel Spectrograms to match Phase 1 training data.
        y, sr = librosa.load(fpath, sr=16000)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
        log_mel = librosa.power_to_db(mel, ref=np.max)
        
        H, W = log_mel.shape
        scaled_mel = scaler.transform(log_mel.reshape(1, H*W)).reshape(H, W)
        
        patches = [scaled_mel[:, s:s+64] for s in range(0, W - 64 + 1, 32)]
        if not patches: continue
            
        batch_t = torch.tensor(np.array(patches), dtype=torch.float32).unsqueeze(1).to(device)
        
        with torch.no_grad():
            out = model(batch_t)
            patch_errors = criterion(out, batch_t).mean(dim=(1,2,3)).cpu().numpy()
            
        # MAX POOLING SCORING
        file_score = np.max(patch_errors)
        labels.append(is_anomaly)
        scores.append(file_score)
        
        # SAVE RECONSTRUCTION VISUAL (First Anomaly)
        if is_anomaly == 1 and not saved_visual and LATEST_RUN_DIR:
            orig_img = batch_t[0,0].cpu().numpy()
            recon_img = out[0,0].cpu().numpy()
            err_map = np.abs(orig_img - recon_img)
            
            fig, ax = plt.subplots(1, 3, figsize=(15, 4))
            ax[0].imshow(orig_img, aspect='auto', origin='lower', cmap='viridis'); ax[0].set_title("Original")
            ax[1].imshow(recon_img, aspect='auto', origin='lower', cmap='viridis'); ax[1].set_title("Reconstruction")
            img3 = ax[2].imshow(err_map, aspect='auto', origin='lower', cmap='magma'); ax[2].set_title("L1 Error")
            plt.colorbar(img3, ax=ax[2])
            plt.savefig(os.path.join(LATEST_RUN_DIR, f"{machine}_anomaly_vision.png"))
            plt.close()
            saved_visual = True

    if labels:
        auc = roc_auc_score(labels, scores)
        pauc = compute_pauc(labels, scores)
        print(f"  -> AUC: {auc:.4f} | pAUC: {pauc:.4f}")
        
        # SAVE HISTOGRAM
        if LATEST_RUN_DIR:
            plt.figure(figsize=(6,4))
            plt.hist(np.array(scores)[np.array(labels)==0], alpha=0.5, label='Normal', color='blue', density=True)
            plt.hist(np.array(scores)[np.array(labels)==1], alpha=0.5, label='Anomaly', color='red', density=True)
            plt.title(f"{machine} Score Distribution")
            plt.legend()
            plt.savefig(os.path.join(LATEST_RUN_DIR, f"{machine}_histogram.png"))
            plt.close()

Saving visuals to: experiments\runs\run_20260423_091243

EVALUATING: ToyCar
  -> AUC: 0.4917 | pAUC: 0.0300

EVALUATING: ToyTrain
  -> AUC: 0.5921 | pAUC: 0.0560

EVALUATING: bearing
  -> AUC: 0.5938 | pAUC: 0.0130

EVALUATING: fan
  -> AUC: 0.5829 | pAUC: 0.1590

EVALUATING: gearbox
  -> AUC: 0.5462 | pAUC: 0.0690

EVALUATING: slider
  -> AUC: 0.5543 | pAUC: 0.0700

EVALUATING: valve
  -> AUC: 0.4097 | pAUC: 0.0630
